均方误差MSE 损失函数

In [1]:
import numpy as np
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

交叉熵Cross Entropy 损失函数

In [ ]:
import numpy as np

# 二分类
def binary_cross_entropy(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -(y_true * np.log(y_pred) + 
             (1 - y_true) * np.log(1 - y_pred))
    return np.mean(loss)

# 多分类
def cross_entropy(y_true, logits):
    exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp / np.sum(exp, axis=1, keepdims=True)
    
    log_probs = -np.log(probs + 1e-12)
    loss = np.sum(y_true * log_probs, axis=1)
    return np.mean(loss)

In [ ]:
import torch
import torch.nn.functional as F

# 二分类
# logits 版本（推荐）
def bce_with_logits(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)

# 如果已经过 sigmoid
def bce(probs, targets):
    return F.binary_cross_entropy(probs, targets)

# 多分类
def multiclass_ce(logits, targets):
    # targets 是类别索引
    return F.cross_entropy(logits, targets)

Hinge Loss 适用于SVM

In [ ]:
# 二分类
def hinge_loss(y_true, y_pred):
    # y_true ∈ {-1,1}
    return np.mean(np.maximum(0, 1 - y_true * y_pred))

# 多分类
def multiclass_hinge_loss(logits, targets):
    margins = logits - logits[np.arange(len(logits)), targets][:, None] + 1
    margins[np.arange(len(logits)), targets] = 0
    return np.mean(np.sum(np.maximum(0, margins), axis=1))

KL散度 适用于知识蒸馏

In [ ]:
def kl_divergence(student_logits, teacher_probs, T=1.0):
    log_probs = F.log_softmax(student_logits / T, dim=1)
    return F.kl_div(log_probs, teacher_probs, reduction='batchmean') * (T * T)

Focal Loss 适用于类别不平衡

In [ ]:
class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)

        if self.alpha is not None:
            at = self.alpha[targets]
            ce_loss = at * ce_loss

        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()